# Project — Build a RAG System from Scratch (Docs-First)

**Goal:** Build a working Retrieval-Augmented Generation (RAG) pipeline that runs entirely on your local machine, then experiment with each component to understand how it works and why it matters.

**Where this fits in the LLM lifecycle:**
1. **Pretraining** gives broad language ability.
2. **Fine-tuning (SFT/RLHF)** shapes behavior and style.
3. **RAG** injects domain knowledge at query time.
4. **Prompting/decoding** controls how answers are produced.

**What you'll use:**
- `sentence-transformers` — to turn text into vectors (embedding model, LLM1)
- `numpy` — as a simple vector store
- `transformers` — to run a local generation model (`flan-t5-base`, LLM2)

**By the end, you should be able to explain:**
- why chunk size has no single "best" value (the chunking paradox)
- why LLM1 and LLM2 are often different models
- how zero-shot, few-shot, and reasoning-style prompts change outputs

**Rules:**
- Everything runs locally. No API keys, no cloud services.
- The knowledge base uses **fictional data the model has never seen**.
- When you get stuck, use Hugging Face docs first and cite the page you used.

---


## Setup

Run the cell below to install the required packages. This may take a few minutes the first time.

**Storage note:**
- `google/flan-t5-base` is ~990MB
- `sentence-transformers/all-MiniLM-L6-v2` is ~80MB

After first download, models are cached locally.

**Hardware note (college laptop friendly):**
- CPU is fine for this project.
- First generation call may be slow; later calls are usually faster.


In [ ]:
# Run this once — installs the packages you need
!pip install sentence-transformers transformers torch umap-learn matplotlib scikit-learn --quiet

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer
import textwrap

# Helper to print long text neatly
def print_wrapped(text, width=90):
    print(textwrap.fill(text, width=width))

---
## Section 0: Hugging Face Warm-Up (Docs First)

**Rule for this section:** do **not** copy code from this notebook. Use official Hugging Face docs/model pages.

Tasks:
1. Load tokenizer + model for **google/flan-t5-base**
2. Generate an answer for: *"Explain RAG in one sentence."*
3. Compare decoding modes:
   - greedy decoding (`do_sample=False`)
   - sampling (`do_sample=True`, `temperature=0.8`)

**What to submit here (short):**
- 2–4 sentences on what changed across decoding modes
- at least one doc URL you used

Helpful docs (start here):
- Transformers text generation: https://huggingface.co/docs/transformers/en/main_classes/text_generation
- Generation strategies: https://huggingface.co/docs/transformers/en/generation_strategies
- Prompting guide: https://huggingface.co/docs/transformers/en/tasks/prompting
- FLAN-T5 model card: https://huggingface.co/google/flan-t5-base

Optional research reading (prompting foundations):
- Chain-of-Thought Prompting Elicits Reasoning in Large Language Models (Wei et al., 2022): https://arxiv.org/abs/2201.11903
- Self-Consistency Improves Chain of Thought Reasoning in Language Models (Wang et al., 2022): https://arxiv.org/abs/2203.11171
- Large Language Models are Zero-Shot Reasoners (Kojima et al., 2022): https://arxiv.org/abs/2205.11916



In [ ]:
# TODO (REFER TO HUGGINGFACE DOCS):
# 1) Load tokenizer + model for google/flan-t5-base
# 2) Tokenize a prompt
# 3) Call model.generate(...) at least twice:
#    - Greedy decoding (no sampling)
#    - Sampling with temperature (do_sample=True)

# Your code should print BOTH outputs so you can compare them.

# Tip: flan-t5 is a Seq2Seq model, so AutoModelForSeq2SeqLM is often used.
# Tip: start small: max_new_tokens=50



**What changed across decoding modes? Doc URL(s) you used:**



---
## Before you start: The Library Analogy

A RAG system works like a research librarian:

1. **You ask a question** (the query)
2. **The librarian searches the shelves** (retrieval) — finds the most relevant books and pages
3. **The librarian reads the relevant pages and gives you a summary** (generation)

The librarian doesn't memorize every book. They know *where to look* and *how to synthesize* what they find.

Here's how the analogy maps to what you'll build:

| Library concept | RAG concept | What you'll use |
|---|---|---|
| The books | Your documents / text | Raw text (strings) |
| Pages / sections | Chunks | Python string splitting |
| The card catalog | Vector index | numpy arrays |
| The catalog's filing system | Embeddings | `sentence-transformers` (LLM₁) |
| The librarian's brain | Generation model | `flan-t5-base` (LLM₂) |

Notice: **the model that creates the index (LLM₁) is NOT the same model that generates the answer (LLM₂).** This is a key design choice you'll explore.

---

### The pipeline you're building

```
Documents → Chunk → Embed (LLM₁) → Store vectors
                                         ↓
Query → Embed (LLM₁) → Find similar chunks → Build prompt → Generate answer (LLM₂)
```

You'll build the top row first (indexing), then the bottom row (querying).

---
## Section 1: Build Your Knowledge Base

We're going to use information about a **fictional company** that doesn't exist anywhere on the internet. This is intentional — if the model gets a question right, it's because RAG worked. If it gets it wrong or hallucinates, it's because it didn't have the right context.

### 1A — The raw documents

Below is the knowledge base. Read through it — you'll be asking questions about this later.

In [ ]:
documents = [
    """NovaMind Technologies was founded in 2019 by Dr. Priya Chakraborty and Marcus Okonkwo 
    in Waterloo, Ontario. The company specializes in AI-powered diagnostic tools for 
    veterinary medicine. Their flagship product, PetScan Pro, uses computer vision to 
    analyze X-rays and MRI scans of animals, detecting fractures, tumors, and joint 
    abnormalities with 94.2% accuracy. PetScan Pro is currently used in over 350 
    veterinary clinics across Canada and the United States.""",
    
    """NovaMind raised a $12 million Series A round in March 2021, led by Horizon Ventures 
    with participation from BioTech Capital and the Ontario Innovation Fund. A subsequent 
    $38 million Series B closed in January 2023, led by Greenfield Partners. The company 
    has been cash-flow positive since Q3 2023. Total headcount as of 2024 is 127 employees, 
    with 74 based in Waterloo and 53 working remotely across North America.""",
    
    """PetScan Pro works by first preprocessing the uploaded medical image using a custom 
    denoising algorithm called ClearVet. The cleaned image is then passed through a 
    fine-tuned ResNet-152 backbone for feature extraction. Classification is handled by a 
    proprietary ensemble of three models: a vision transformer (ViT-L), a convolutional 
    network, and a lightweight decision tree that acts as a tiebreaker. Results are returned 
    in under 8 seconds with a detailed heatmap overlay showing areas of concern.""",
    
    """In September 2024, NovaMind announced FeatherCheck, a new product designed for avian 
    diagnostics. FeatherCheck analyzes feather samples and blood work photographs to screen 
    for common diseases in parrots, eagles, and poultry. Early trials at the University of 
    Guelph showed 87.5% accuracy in detecting Psittacine Beak and Feather Disease (PBFD) 
    from feather images alone. FeatherCheck is expected to launch commercially in Q2 2025.""",
    
    """Dr. Priya Chakraborty holds a PhD in Biomedical Engineering from the University of 
    Toronto and previously worked at Google Health for three years. Marcus Okonkwo has a 
    Masters in Computer Science from the University of Waterloo and was a founding engineer 
    at Shopify's machine learning division. The two met at a veterinary AI hackathon in 2018 
    and decided to start NovaMind after winning first place with a prototype that could 
    detect hip dysplasia in dogs from smartphone photos.""",
    
    """NovaMind's engineering team follows a rigorous model evaluation protocol. Every model 
    update must pass a 4-stage pipeline: unit tests on synthetic data, validation on a held-out 
    dataset of 15,000 annotated images, a red-team adversarial review by the safety team, and 
    finally a 2-week shadow deployment where the new model runs alongside the production model 
    and outputs are compared. Only if the new model matches or exceeds production accuracy with 
    no increase in false negatives is it promoted to production.""",
    
    """The company's data infrastructure runs on a hybrid cloud setup. Training workloads use 
    a cluster of 8 NVIDIA A100 GPUs hosted on AWS, while inference for PetScan Pro runs on 
    edge devices — specifically NVIDIA Jetson Orin modules deployed directly in partner clinics. 
    This edge deployment ensures that sensitive animal medical data never leaves the clinic's 
    network, addressing privacy concerns raised by several provincial veterinary boards.""",
    
    """NovaMind's main competitors include VetAI (based in San Francisco, raised $45M, focused 
    on canine diagnostics only), AnimalSense (London-based, uses ultrasound analysis), and 
    PawPath Diagnostics (a subsidiary of a large pharmaceutical company). NovaMind differentiates 
    itself through multi-species support, edge deployment for data privacy, and their ensemble 
    architecture which consistently outperforms single-model approaches in independent benchmarks 
    published by the Veterinary AI Consortium."""
]

print(f"Knowledge base: {len(documents)} documents")
print(f"Total characters: {sum(len(d) for d in documents):,}")

### 1B — Chunking

Right now each document is one big block of text. In a real RAG system, you split documents into smaller **chunks** so that retrieval can be more precise — you want to find the *specific paragraph* that answers a question, not an entire 10-page document.

But here's the catch: **there's no perfect chunk size.**
- Too small → you lose context (a sentence fragment doesn't make sense on its own)
- Too big → you dilute the signal (the relevant sentence is buried in noise)

For now, we'll use a simple approach: split each document into chunks of roughly `chunk_size` characters, with some overlap so we don't cut sentences in half.

**Build the chunking function below.**

In [ ]:
def chunk_text(text, chunk_size=200, overlap=50):
    """Split `text` into overlapping chunks.

    Docs goal: you should be able to explain why overlap exists and how chunk_size affects retrieval.

    Requirements:
    - Return a list[str]
    - Chunks should be non-empty
    - Use overlap so important info near boundaries isn't lost

    Hint:
    Use a sliding window over the text. At each step, take a slice from `start` to `end` (where `end = start + chunk_size`),
    append it to the list, then increment `start` by `chunk_size - overlap` until you reach the end of the text.
    Be careful to handle the last slice so you don't drop any trailing characters.
    """
    # TODO: implement chunking
    return []


In [ ]:
# Chunk all documents using your chunker.
# The variable name `all_chunks` is used later in the notebook.
all_chunks = []
chunk_sources = []  # new: track which document each chunk came from
for doc_idx, doc in enumerate(documents):
    chunks = chunk_text(doc, chunk_size=200, overlap=50)
    all_chunks.extend(chunks)
    chunk_sources.extend([doc_idx] * len(chunks))

print(f"Total chunks: {len(all_chunks)}")
if len(all_chunks) > 0:
    print()
    print("Example chunk:")
    print(all_chunks[0])
else:
    print("No chunks yet — implement `chunk_text` above.")


**Checkpoint:** Look at a few of your chunks. Do they make sense as standalone pieces of text? Could someone read one chunk and get useful information from it? We'll come back to this question in the experiments section.

In [ ]:
# Run after you implement `chunk_text` above.
# This cell is safe to run even if you haven't implemented it yet.

if len(all_chunks) == 0:
    print("No chunks to show yet.")
else:
    for i, chunk in enumerate(all_chunks[:5]):
        print(f"--- Chunk {i} ({len(chunk)} chars) ---")
        print(chunk)
        print()


---
## Section 2: Create Your Index (Embeddings + Vector Store)

Now we need to turn each chunk into a **vector** (a list of numbers) so we can mathematically compare chunks to queries.

This is the **LLM₁** in the pipeline — the embedding model. It doesn't generate text; it just converts text into a numerical fingerprint that captures meaning.

### 2A — Load the embedding model

In [ ]:
# TODO (docs-first): load a sentence-transformers embedding model.
# Required: use the model name below (small + fast on CPU).
# Model: sentence-transformers/all-MiniLM-L6-v2
#
# Goal: create `embedding_model` with an `.encode(...)` method.
#
# Helpful starting points (search these):
# - "SentenceTransformer encode"
# - "sentence-transformers all-MiniLM-L6-v2"

embedding_model = None  # replace with your loaded model

# Quick sanity check (should print a vector shape like (384,))
if embedding_model is None:
    print("embedding_model is None — load it using sentence-transformers docs.")
else:
    test_embedding = embedding_model.encode("This is a test sentence.")
    print("Embedding shape:", np.array(test_embedding).shape)


Each piece of text becomes a vector of 384 numbers. These numbers don't mean anything individually — but texts with **similar meaning** will have vectors that **point in a similar direction.**

### 2B — Embed all chunks and build the vector store

Our "vector store" is just a numpy array. Each row is one chunk's embedding.

In [ ]:
# TODO: embed every chunk to create your "vector index"
# Required variable name: `chunk_embeddings` (shape: [num_chunks, embedding_dim])

chunk_embeddings = None

if embedding_model is None:
    print("Load embedding_model first.")
elif len(all_chunks) == 0:
    print("Create chunks first (all_chunks is empty).")
else:
    chunk_embeddings = embedding_model.encode(all_chunks)
    chunk_embeddings = np.array(chunk_embeddings)
    print(f"Vector store shape: {chunk_embeddings.shape}")
    print(f"That's {chunk_embeddings.shape[0]} chunks, each represented by {chunk_embeddings.shape[1]} numbers.")


### 2C — Quick test: semantic similarity

Before building retrieval, let's verify that the embeddings actually capture meaning. We'll compare three sentences and check which two are most similar.

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors using numpy.

    Returns a float in [-1, 1]. If either vector has zero magnitude, returns 0.0.
    """

# TODO: Create your own three test sentences
sentences = [

]

if embedding_model is None:
    print("Load embedding_model first.")
else:
    vecs = embedding_model.encode(sentences)
    vecs = np.array(vecs)

    sim01 = cosine_similarity(vecs[0], vecs[1])
    sim02 = cosine_similarity(vecs[0], vecs[2])
    sim12 = cosine_similarity(vecs[1], vecs[2])

    print("Similarity(0,1):", sim01)
    print("Similarity(0,2):", sim02)
    print("Similarity(1,2):", sim12)

    print("\nExpected: (0,1) should be the highest similarity.")


**Checkpoint:** The similar sentences should have a much higher score than the dissimilar ones. If this works, your embeddings are capturing meaning correctly.

### UMAP Visualization (optional)

To make semantic similarity tangible, we'll reduce the high-dimensional chunk embeddings to two dimensions using UMAP.
Each point represents a chunk, colored by the document it came from. We'll also embed a sample query and plot it as a red star.
Points that are close together in this 2D space should correspond to semantically similar text.


> **Note:** If this cell throws an installation error related to C++ or numba (common on some Windows machines), you can skip this cell. It will not affect the rest of the project.

In [ ]:
# UMAP visualization
# This cell fits a UMAP model on your chunk embeddings and plots them.
# Requires that you have already built `chunk_embeddings` and `chunk_sources`.
if 'chunk_embeddings' not in globals() or chunk_embeddings is None or len(chunk_embeddings) == 0:
    print('Build your chunk embeddings first (Section 2).')
else:
    import umap.umap_ as umap
    import matplotlib.pyplot as plt
    import numpy as np

    # Fit UMAP on the chunk embeddings (we use cosine distance)
    reducer = umap.UMAP(random_state=42, metric='cosine')
    embedding_2d = reducer.fit_transform(chunk_embeddings)

    # Plot all chunks color-coded by source document if available
    plt.figure(figsize=(8,6))
    if 'chunk_sources' in globals():
        unique_docs = sorted(set(chunk_sources))
        for doc_idx in unique_docs:
            idxs = [i for i, d in enumerate(chunk_sources) if d == doc_idx]
            xs = embedding_2d[idxs, 0]
            ys = embedding_2d[idxs, 1]
            plt.scatter(xs, ys, label=f'Doc {doc_idx+1}', alpha=0.7)
    else:
        plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1])
    plt.legend()
    plt.title('UMAP projection of chunk embeddings')
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.show()

    # Embed a query and project it onto the same UMAP model
    query = 'Who founded NovaMind?'
    if 'embedding_model' in globals() and embedding_model is not None:
        query_vec = embedding_model.encode([query])
        query_2d = reducer.transform(query_vec)
        plt.figure(figsize=(8,6))
        if 'chunk_sources' in globals():
            for doc_idx in unique_docs:
                idxs = [i for i, d in enumerate(chunk_sources) if d == doc_idx]
                xs = embedding_2d[idxs, 0]
                ys = embedding_2d[idxs, 1]
                plt.scatter(xs, ys, label=f'Doc {doc_idx+1}', alpha=0.7)
        else:
            plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1])
        plt.scatter(query_2d[0,0], query_2d[0,1], marker='*', s=200, color='red', label='Query')
        plt.legend()
        plt.title('UMAP projection with query')
        plt.xlabel('UMAP 1')
        plt.ylabel('UMAP 2')
        plt.show()
    else:
        print('Build the embedding model first.')


---
## Section 3: Retrieval

Now we build the "librarian's search" — given a query, find the most relevant chunks.

The process:
1. Embed the query using the **same embedding model** (LLM₁)
2. Compute cosine similarity between the query vector and **every** chunk vector
3. Return the top-k most similar chunks

### 3A — Build the retrieval function

In [ ]:
def retrieve(query, chunks, chunk_embeddings, embedding_model, top_k=3):
    """Return the top_k most relevant chunks for `query`.

    Inputs:
      - query: str
      - chunks: list[str]
      - chunk_embeddings: np.ndarray shape (N, D)
      - embedding_model: has .encode(...)
      - top_k: int

    Output:
      - list of tuples: (chunk_text, similarity_score, chunk_index)

    Requirements:
      1) Embed the query with the SAME embedding model used for chunks
      2) Compute cosine similarity vs every chunk
      3) Return top_k chunks sorted by similarity (highest first)
      4) Include the chunk index so you can cite sources later
    """
    # TODO: implement retrieval
    return []

# Safety check
if chunk_embeddings is None:
    print("Build chunk_embeddings first.")
else:
    print("Ready to retrieve. (Implement `retrieve` above.)")


### 3B — Test retrieval

Try a few queries and see what chunks come back. Do they make sense?

In [ ]:
query = "Who founded NovaMind?"

results = retrieve(query, all_chunks, chunk_embeddings, embedding_model, top_k=3)

print(f"Query: '{query}'\n")
for i, (chunk, score, chunk_id) in enumerate(results):
    print(f"Result {i+1} [chunk {chunk_id}] (similarity: {score:.4f}):")
    print_wrapped(chunk)
    print()

In [ ]:
# Try another query
query = "How does PetScan Pro process images?"

results = retrieve(query, all_chunks, chunk_embeddings, embedding_model, top_k=3)

print(f"Query: '{query}'\n")
for i, (chunk, score, chunk_id) in enumerate(results):
    print(f"Result {i+1} [chunk {chunk_id}] (similarity: {score:.4f}):")
    print_wrapped(chunk)
    print()

**Checkpoint:** The top result should be the chunk that actually answers your question. If chunk #1 is irrelevant but chunk #3 is perfect, that's a signal your chunking or embedding might need adjustment — something you'll explore later.

---
## Section 4: Generation

Now we load the **generation model** (LLM₂). This is a completely different model from the embedding model. Its job: read the retrieved context and the question, then produce an answer.

**Heads up:** `flan-t5-base` is small (~990MB). It will produce functional but sometimes clunky answers. That's fine — you're learning the *architecture*, not building a product. In production you'd swap this for a much larger model.

### 4A — Load the generation model (docs-first)

**Required baseline model:** `google/flan-t5-base`

Requirements:
- Create `tokenizer`
- Create `gen_model`

Helpful starting points (search these):
- "T5Tokenizer from_pretrained"
- "T5ForConditionalGeneration from_pretrained"
- "AutoTokenizer AutoModelForSeq2SeqLM flan-t5"

Optional local upgrade ideas (after baseline works):
- `Qwen/Qwen2.5-1.5B-Instruct` (stronger, heavier)
- `microsoft/Phi-3.5-mini-instruct` (stronger, heavier)
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (lighter, weaker)

If you switch models, document what changed in prompt format and answer quality.

In [ ]:
# TODO (docs-first): load tokenizer + generation model for google/flan-t5-base
# Required variable names: `tokenizer` and `gen_model`

tokenizer = None   # replace
gen_model = None   # replace

if tokenizer is None or gen_model is None:
    print("tokenizer and/or gen_model is None — load them using HF docs.")
else:
    print("Generation model loaded.")


### 4B — Build the prompt

This is the critical step. The **prompt** is what the generation model actually sees. It combines:
- The retrieved context (from Section 3)
- The user's question
- Instructions telling the model how to behave

The model isn't "looking things up" — you are **physically placing the relevant information into its input** and asking it to synthesize an answer. This is what the whiteboard notes mean by "prompt + context extracted."

In [ ]:
def build_prompt(query, retrieved_chunks):
    """Assemble the final prompt from the question and retrieved context.

    retrieved_chunks: list of (chunk_text, similarity_score, chunk_index)

    Requirements:
    - Include clear separators between chunks
    - Include chunk IDs so the model can cite them (e.g., [chunk_12])
    - Add an instruction: answer using ONLY the context; if missing, say you don't know.
    """
    # TODO: implement prompt building
    return f"Question: {query}\n\nContext:\n\n(implement build_prompt)\n"

# Quick print to see what the prompt looks like
demo_query = "What does NovaMind sell?"
if embedding_model is None or chunk_embeddings is None or len(all_chunks) == 0:
    print("Build your index first.")
else:
    demo_retrieved = retrieve(demo_query, all_chunks, chunk_embeddings, embedding_model, top_k=3)
    prompt = build_prompt(demo_query, demo_retrieved)
    print("=== THE FULL PROMPT SENT TO THE MODEL ===")
    print(prompt[:1200], "..." if len(prompt) > 1200 else "")
    print("=========================================")

**Read the prompt above carefully.** This is exactly what the generation model receives. The model doesn't "know" about NovaMind — it's reading the context you retrieved and placed there, just like a student reading an open-book exam.

### 4C — Generate an answer

***

**Model note:** `flan-t5-base` is efficient but often gives very short, direct answers rather than conversational paragraphs. As long as the facts are correct, your pipeline is working.


In [ ]:
def generate_answer(prompt, tokenizer, model, max_new_tokens=200, temperature=0.7, do_sample=True):
    """Generate an answer using a local Hugging Face model.

    Docs-first requirement: implement generation using `transformers` APIs.
    For flan-t5 (seq2seq), typical steps:
      1) tokenize -> input_ids
      2) model.generate(...)
      3) decode tokens -> string

    Requirements:
    - Support greedy decoding when do_sample=False
    - Support sampling with temperature when do_sample=True
    """
    # TODO: implement using tokenizer + model.generate + tokenizer.decode
    return "(implement generate_answer)"

print("generate_answer stub ready (implement it using HF docs).")


---
## Section 5: The Full RAG Pipeline

Now wrap everything into one function: query in → answer out.

### 5A — Build the pipeline

In [ ]:
def rag_query(question, chunks, chunk_embeddings, embedding_model, tokenizer, gen_model,
              top_k=3, temperature=0.7, show_context=False):
    """Full RAG pipeline: question -> retrieved context -> prompt -> generated answer.

    Required steps:
      1) retrieved = retrieve(...)
      2) prompt = build_prompt(...)
      3) answer = generate_answer(...)

    Return:
      - answer (str)
      - optionally print retrieved context if show_context=True
    """
    # TODO: implement the pipeline
    return "(implement rag_query)"

print("rag_query stub ready (implement it after retrieve/build_prompt/generate_answer).")


### 5B — Try it out

Ask several questions about NovaMind. Use `show_context=True` so you can see what the retriever found.

In [ ]:
questions = [
    "Who founded NovaMind and where?",
    "How much funding has NovaMind raised in total?",
    "What is FeatherCheck?",
    "How does NovaMind handle data privacy?",
    "Who are NovaMind's competitors?"
]

if tokenizer is None or gen_model is None:
    print("Load tokenizer + gen_model first (Section 4A).")
elif embedding_model is None or chunk_embeddings is None or len(all_chunks) == 0:
    print("Missing pieces: build chunks + embeddings first.")
else:
    print("=== WITH RAG ===\n")
    for q in questions:
        ans = rag_query(q, all_chunks, chunk_embeddings, embedding_model, tokenizer, gen_model,
                        top_k=3, temperature=0.7, show_context=True)
        print(f"Q: {q}")
        print(f"A: {ans}")
        print("=" * 80)
        print()

### 5C — Without RAG (the control test)

Now ask the **same questions** directly to the model, **without** any retrieved context. This shows you exactly what RAG is adding.

In [ ]:
print("=== WITHOUT RAG (control) ===\n")

if tokenizer is None or gen_model is None:
    print("Load tokenizer + gen_model first (Section 4A).")
else:
    for q in questions:
        bare_prompt = f"Answer the following question. If you don't know, say you don't know.\n\nQuestion: {q}\nAnswer:"
        ans = generate_answer(bare_prompt, tokenizer, gen_model, max_new_tokens=120, temperature=0.7, do_sample=True)
        print(f"Q: {q}")
        print(f"A: {ans}")
        print()


**Checkpoint:** The model without RAG should either hallucinate (make things up) or give generic non-answers, because NovaMind is fictional. With RAG, it should give accurate answers grounded in the retrieved chunks. This is the entire point of RAG.

---
## Section 6: Experiments

Now that the pipeline works, it's time to **break it, tune it, and understand it.** Each experiment targets a specific component.


In [ ]:
# Experiment 1 — your code here


**Your findings:**

- chunk_size=50: 
- chunk_size=200: 
- chunk_size=500: 
- Which worked best and why?



### Experiment 2: Top-K (How many chunks to retrieve)

Using the default chunk_size=200, try the same question with `top_k=1`, `top_k=3`, and `top_k=10`.

Think about:
- More chunks = more context, but also more noise
- Fewer chunks = more focused, but might miss relevant info
- Also: the model has a limited input window (512 tokens for flan-t5-base). What happens when you retrieve too many chunks?

In [ ]:
# Experiment 2 — your code here


**Your findings:**

- top_k=1: 
- top_k=3: 
- top_k=10: 
- What's the tradeoff?



### Experiment 3: Prompting Techniques (Zero-shot vs Few-shot)

Use the **same question** and **same retrieved chunks** for all versions so the comparison is fair.

Test these prompt styles:
1. **Zero-shot**: direct instruction + context
2. **Few-shot**: add 1–2 example Q/A pairs before the real question

(Optional) If you're curious, you can try more elaborate prompting strategies (like asking for a brief evidence summary before the final answer), but these tend to produce longer outputs without improving factuality on small models.

Important nuance:
- Prompting can change the structure of the answer but doesn't fix missing information.
- Keep your examples concise to avoid pushing relevant context out of the model's input window.

Score each output with this mini-rubric:
- Correctness (0–2)
- Grounded in retrieved context? (Yes/No)
- Cites chunk IDs? (Yes/No)
- Concise enough for a user? (Yes/No)

Helpful docs:
- Prompting guide: https://huggingface.co/docs/transformers/en/tasks/prompting
- Chat templating (if using chat models): https://huggingface.co/docs/transformers/en/chat_templating

Research links (optional reading):
- Chain-of-Thought prompting: https://arxiv.org/abs/2201.11903
- Self-consistency for reasoning: https://arxiv.org/abs/2203.11171
- Zero-shot CoT: https://arxiv.org/abs/2205.11916


In [ ]:
# Experiment 3 — your code here


**Your findings:**

- Zero-shot: 
- Few-shot: 
- Which worked best? Did any prompt style hurt quality?



---
## Section 7: Reflection

Answer in your own words. These check understanding of *why* each component exists.

### 7A — The pipeline

For each component below, answer: **What goes in? What comes out? Why can't we skip it?**

1. **Chunking** — what goes in, what comes out, and what goes wrong if you skip it?
2. **Embedding model (LLM1)** — what goes in, what comes out, and why not keyword matching only?
3. **Vector store + similarity search** — what goes in, what comes out, and what if retrieval returns random chunks?
4. **Generation model (LLM2)** — what goes in, what comes out, and why not just return raw chunks?
5. **Prompt template** — what changed in Experiment 3 when you changed prompt style?
6. **Prompting vs training** — what can prompting change that fine-tuning would change more reliably?


**Your answers:**

1. Chunking: 

2. Embedding model: 

3. Vector store: 

4. Generation model: 

5. Prompt template: 

6. Prompting vs training: 


### 7B — LLM1 vs LLM2

You used different models for different jobs:
- embeddings (`all-MiniLM-L6-v2`) for retrieval
- generation (`flan-t5-base`) for answers

1. Why is it often better to separate retrieval and generation models?
2. If you had budget to upgrade only one first, which would you choose and why?
3. What tradeoff did you observe between speed and answer quality on your machine?


**Your answers:**

1. 

2. 

3. 


### 7C — RAG vs Fine-Tuning

Two ways to give a model specialized capability:
- **RAG** — keep model weights fixed; retrieve documents at query time
- **Fine-tuning** — update model weights using task/domain data

Based on what you built:
1. One advantage of RAG over fine-tuning.
2. One advantage of fine-tuning over RAG.
3. If NovaMind docs update weekly, which approach is more practical and why?
4. If you needed a very consistent brand tone, would RAG alone be enough? Why/why not?


**Your answers:**

1. 

2. 

3. 

4. 


### 7D — Limitations and Simplification

You probably noticed `flan-t5-base` is useful but limited. That's intentional for learning architecture.

1. Name two specific failure modes you observed (wrong facts, truncation, weak phrasing, etc.) and likely causes.
2. For each failure, is the fix mainly in **retrieval**, **prompting**, or **generation model**?
3. If you had to simplify this project for a 90-minute lab, what would you remove and what would you keep?


**Your answers:**

1. 

2. 

3. 


---
## References

Prompting papers referenced in this notebook:

1. Wei, J. et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. https://arxiv.org/abs/2201.11903
2. Wang, X. et al. (2022). *Self-Consistency Improves Chain of Thought Reasoning in Language Models*. https://arxiv.org/abs/2203.11171
3. Kojima, T. et al. (2022). *Large Language Models are Zero-Shot Reasoners*. https://arxiv.org/abs/2205.11916

Use these as conceptual grounding for Experiment 3. Your grade is based on your own implementation and analysis, not reproducing paper results.
